## Cell 1


In [ ]:
!pip install -q \
llama-index-core \
llama-index-readers-file \
llama-index-llms-ollama \
llama-index-embeddings-ollama

## Cell 2


In [ ]:
from llama_index.core import Settings
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

# Language model
Settings.llm = Ollama(
    model="qwen2.5:0.5b-instruct",
    base_url="http://localhost:11434",
    request_timeout=180.0,
    context_window=4096
)

# Embedding model
Settings.embed_model = OllamaEmbedding(
    model_name="all-minilm",
    base_url="http://localhost:11434"
)

# Chunking settings
Settings.chunk_size = 512
Settings.chunk_overlap = 80

print("Ollama models connected to LlamaIndex!")

## Cell 3


In [ ]:
from google.colab import files
from llama_index.core import SimpleDirectoryReader

uploaded = files.upload()

pdf_path = next(iter(uploaded))

documents = SimpleDirectoryReader(
    input_files=[pdf_path]
).load_data()

print("Document loaded successfully!")
print("Total document sections:", len(documents))

## Cell 4


In [ ]:
import requests

try:
    response = requests.get(
        "http://localhost:11434",
        timeout=5
    )
    print(response.text)

except Exception as e:
    print("Ollama is not responding:", e)

## Cell 5


In [ ]:
import subprocess
import time
import requests

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

for i in range(30):
    try:
        response = requests.get(
            "http://localhost:11434",
            timeout=2
        )

        if response.status_code == 200:
            print("Ollama server restarted successfully!")
            break

    except requests.exceptions.RequestException:
        time.sleep(2)

else:
    raise RuntimeError("Ollama failed to start.")

## Cell 6


In [ ]:
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex.from_documents(
    documents,
    show_progress=True
)

print("RAG index created successfully!")

## Cell 7


In [ ]:
query_engine = index.as_query_engine(
    similarity_top_k=3,
    response_mode="compact"
)

print("Your research assistant is ready!")

## Cell 8


In [ ]:
def ask_question(question):

    response = query_engine.query(question)

    print("\nANSWER:\n")
    print(response)

    print("\nRETRIEVED SOURCES:\n")

    for i, source in enumerate(response.source_nodes):

        print("Source:", i + 1)
        print("Metadata:", source.node.metadata)
        print("Relevance score:", source.score)
        print("Text:", source.node.text[:400])
        print("-" * 50)

## Cell 9


In [ ]:
ask_question(
    "What is the Transformer architecture?"
)